# WFP Location-Routing Experiment Recreated With `scalable_distances`

This notebook recreates the location-routing experiment described in the WFP slide deck in `C:\local\temp\WFP`. The slides describe a Nusa Tenggara Timur school-meal planning workflow that moves from geocoded school records to island-level facility location and then to assigned-school routing diagnostics.

The original slide deck reports country-scale and island-scale results, but the slide folder contains the deck and figures rather than the raw 14,103 school registry rows or the original road-distance matrices. This notebook therefore has two layers:

1. It records the slide-scale reported numbers so the experiment target is explicit.
2. It recreates the same optimization mechanics on a small, executable Sabu-style road network using the new `scalable_distances` routing contracts.

The miniature experiment is not a replacement for the original 303-school Sabu run. It is a runnable template for the same workflow: routing-target validation, uncapacitated facility location, assignment, TSP routing, and fallback diagnostics for disconnected road components.

In [ ]:
from itertools import combinations
from functools import lru_cache
from pathlib import Path
import math
import sys

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src" / "scalable_distances").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_ROOT = PROJECT_ROOT / "src"
assert (SRC_ROOT / "scalable_distances").exists(), f"Could not find scalable_distances under {PROJECT_ROOT}"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd

from scalable_distances.routing.base import NetworkData
from scalable_distances.routing.strategies import NetworkXRouter

## Slide Facts Used As The Target

The WFP deck describes a milestone experiment for Nusa Tenggara Timur:

- 14,103 source school records.
- 14,101 records geocoded, leaving 2 unresolved.
- 9,400 selected coordinates from open sources before Google fallback.
- 693 direct OSM school-level matches.
- Facility-location models solved independently for 7 island groups.
- Fixed opening cost calibrated as `2 km * island demand`, so another facility must reduce average weighted distance by roughly 2 km.
- Recommended facility-location run selects 27 facilities for 14,099 served schools.
- Sabu routing illustration has two selected facilities serving 107 and 196 schools, with routes of 151.57 km and 258.40 km.
- One Sabu route has four straight-line fallback segments because of disconnected OSM road components.

In [ ]:
geocoding_summary = pd.DataFrame([
    {"metric": "source_records", "slide_value": 14_103},
    {"metric": "geocoded_records", "slide_value": 14_101},
    {"metric": "unresolved_records", "slide_value": 2},
    {"metric": "open_selected_before_google", "slide_value": 9_400},
    {"metric": "direct_osm_school_matches", "slide_value": 693},
])

facility_location_slide_results = pd.DataFrame([
    {"island": "Alor", "schools": 774, "candidates": 112, "open": 3, "mean_km": 10.59, "p90_km": 21.49, "time_s": 2.8},
    {"island": "Flores", "schools": 4426, "candidates": 120, "open": 5, "mean_km": 17.17, "p90_km": 33.02, "time_s": 32.6},
    {"island": "Flores-Lembata", "schools": 1122, "candidates": 120, "open": 4, "mean_km": 10.51, "p90_km": 23.24, "time_s": 2.2},
    {"island": "Rote", "schools": 397, "candidates": 80, "open": 3, "mean_km": 7.59, "p90_km": 13.34, "time_s": 0.4},
    {"island": "Sabu", "schools": 303, "candidates": 70, "open": 2, "mean_km": 7.53, "p90_km": 14.20, "time_s": 0.3},
    {"island": "Sumba", "schools": 2149, "candidates": 120, "open": 5, "mean_km": 13.17, "p90_km": 32.60, "time_s": 8.2},
    {"island": "Timor", "schools": 4928, "candidates": 120, "open": 5, "mean_km": 15.47, "p90_km": 33.38, "time_s": 19.5},
])

sabu_slide_routes = pd.DataFrame([
    {"facility": "Sabu_0033", "schools": 107, "route_km": 151.57, "runtime_s": 2.2},
    {"facility": "Sabu_0046", "schools": 196, "route_km": 258.40, "runtime_s": 173.1},
])

geocoding_summary, facility_location_slide_results, sabu_slide_routes

## Model 1: Island-Level Facility Location

The slide deck uses an uncapacitated facility-location model per island:

\[
\min \sum_{i\in I} f_i y_i + \sum_{j\in J}\sum_{i\in I} w_j d_{ij} x_{ij}
\]

\[
\sum_{i\in I} x_{ij}=1\quad(\forall j),\qquad
x_{ij}\le y_i\quad(\forall i,j),\qquad
x_{ij},y_i\in\{0,1\}.
\]

The miniature run below follows the same contract. Candidate facilities and schools are snapped to a road network. The distance matrix is computed through `NetworkXRouter`. Missing paths are retained with straight-line fallback distances and explicit fallback flags, matching the slide warning that routing diagnostics should reveal disconnected OSM road components.

In [ ]:
def edge_rows(edges):
    rows = []
    for u, v, length_m in edges:
        rows.append({"u": u, "v": v, "length_m": float(length_m)})
        rows.append({"u": v, "v": u, "length_m": float(length_m)})
    return rows

nodes = pd.DataFrame([
    {"node_id": 1, "lon": 0.0, "lat": 0.0},
    {"node_id": 2, "lon": 1.0, "lat": 0.2},
    {"node_id": 3, "lon": 2.0, "lat": 0.3},
    {"node_id": 4, "lon": 3.0, "lat": 0.4},
    {"node_id": 5, "lon": 4.0, "lat": 0.2},
    {"node_id": 6, "lon": 5.0, "lat": 0.0},
    {"node_id": 7, "lon": 6.0, "lat": -0.1},
    {"node_id": 8, "lon": 7.0, "lat": 0.1},
    {"node_id": 9, "lon": 8.0, "lat": 0.0},
    {"node_id": 10, "lon": 9.0, "lat": -0.2},
    {"node_id": 11, "lon": 10.0, "lat": 0.0},
    {"node_id": 12, "lon": 2.0, "lat": -1.0},
    {"node_id": 13, "lon": 3.0, "lat": -1.1},
    {"node_id": 14, "lon": 4.0, "lat": -1.0},
    {"node_id": 15, "lon": 5.0, "lat": -0.8},
    {"node_id": 20, "lon": 8.4, "lat": 1.5},  # disconnected component
    {"node_id": 21, "lon": 8.9, "lat": 1.8},
])

edges = pd.DataFrame(edge_rows([
    (1, 2, 6200), (2, 3, 6100), (3, 4, 6100), (4, 5, 6500), (5, 6, 6200),
    (6, 7, 6100), (7, 8, 6400), (8, 9, 6100), (9, 10, 6200), (10, 11, 6500),
    (3, 12, 7600), (12, 13, 6200), (13, 14, 6100), (14, 15, 6600), (15, 6, 7000),
    (20, 21, 4200),
]))

network = NetworkData(nodes=nodes, edges=edges)

candidates = pd.DataFrame([
    {"source_id": "Sabu_0033", "source_type": "candidate_facility", "lon": 2.05, "lat": 0.32},
    {"source_id": "Sabu_0046", "source_type": "candidate_facility", "lon": 7.05, "lat": 0.08},
    {"source_id": "Sabu_west_alt", "source_type": "candidate_facility", "lon": 0.1, "lat": 0.02},
    {"source_id": "Sabu_south_alt", "source_type": "candidate_facility", "lon": 4.05, "lat": -0.95},
])

schools = pd.DataFrame([
    {"target_id": "SCH_001", "target_type": "school", "lon": 0.05, "lat": 0.02, "demand": 110},
    {"target_id": "SCH_002", "target_type": "school", "lon": 1.05, "lat": 0.22, "demand": 80},
    {"target_id": "SCH_003", "target_type": "school", "lon": 2.1, "lat": 0.28, "demand": 140},
    {"target_id": "SCH_004", "target_type": "school", "lon": 3.1, "lat": 0.38, "demand": 120},
    {"target_id": "SCH_005", "target_type": "school", "lon": 2.0, "lat": -1.05, "demand": 95},
    {"target_id": "SCH_006", "target_type": "school", "lon": 4.1, "lat": -0.98, "demand": 130},
    {"target_id": "SCH_007", "target_type": "school", "lon": 5.05, "lat": -0.82, "demand": 105},
    {"target_id": "SCH_008", "target_type": "school", "lon": 6.05, "lat": -0.08, "demand": 115},
    {"target_id": "SCH_009", "target_type": "school", "lon": 7.05, "lat": 0.12, "demand": 160},
    {"target_id": "SCH_010", "target_type": "school", "lon": 8.05, "lat": 0.02, "demand": 135},
    {"target_id": "SCH_011", "target_type": "school", "lon": 9.05, "lat": -0.18, "demand": 90},
    {"target_id": "SCH_012", "target_type": "school", "lon": 8.85, "lat": 1.78, "demand": 75},
])

router = NetworkXRouter()
router.prepare(network, {})
snapped_candidates = router.snap(candidates)
snapped_schools = router.snap(schools)
road_matrix = router.route_many(snapped_candidates, snapped_schools)

snapped_candidates, snapped_schools.head(), road_matrix.head()

In [ ]:
COORD_SCALE_M = 6000.0
FALLBACK_FACTOR = 1.25

def straight_line_m(a_lon, a_lat, b_lon, b_lat):
    return math.hypot(a_lon - b_lon, a_lat - b_lat) * COORD_SCALE_M * FALLBACK_FACTOR

def complete_facility_school_matrix(candidates, schools, road_matrix):
    road_lookup = {(r.source_id, r.target_id): r.total_dist for r in road_matrix.itertuples(index=False)}
    rows = []
    for fac in candidates.itertuples(index=False):
        for sch in schools.itertuples(index=False):
            road_distance = road_lookup.get((fac.source_id, sch.target_id))
            if road_distance is None:
                distance = straight_line_m(fac.lon, fac.lat, sch.lon, sch.lat)
                fallback = True
                mode = "straight_line_disconnected_component"
            else:
                distance = float(road_distance)
                fallback = False
                mode = "road_network"
            rows.append({
                "source_id": fac.source_id,
                "target_id": sch.target_id,
                "demand": sch.demand,
                "total_dist": distance,
                "fallback": fallback,
                "mode": mode,
            })
    return pd.DataFrame(rows)

facility_school_matrix = complete_facility_school_matrix(candidates, schools, road_matrix)
facility_school_matrix.head(12)

In [ ]:
def solve_uncapacitated_facility_location(matrix, setup_km=2.0):
    source_ids = sorted(matrix["source_id"].unique())
    demand_total = matrix.drop_duplicates("target_id")["demand"].sum()
    fixed_cost = setup_km * 1000.0 * demand_total
    best = None
    evaluated = []
    for r in range(1, len(source_ids) + 1):
        for open_set in combinations(source_ids, r):
            open_set = tuple(open_set)
            subset = matrix[matrix["source_id"].isin(open_set)].copy()
            assignment = subset.sort_values("total_dist").groupby("target_id", as_index=False).first()
            weighted_distance = (assignment["demand"] * assignment["total_dist"]).sum()
            objective = fixed_cost * len(open_set) + weighted_distance
            row = {
                "open_facilities": open_set,
                "n_open": len(open_set),
                "fixed_cost": fixed_cost * len(open_set),
                "weighted_distance": weighted_distance,
                "objective": objective,
                "mean_km": (weighted_distance / assignment["demand"].sum()) / 1000.0,
                "fallback_assignments": int(assignment["fallback"].sum()),
            }
            evaluated.append(row)
            if best is None or objective < best["objective"]:
                best = row | {"assignment": assignment}
    return best, pd.DataFrame(evaluated).sort_values("objective")

ufl_solution, ufl_sensitivity = solve_uncapacitated_facility_location(facility_school_matrix, setup_km=2.0)
assignment = ufl_solution["assignment"].merge(schools, on=["target_id", "demand"], how="left")

ufl_summary = pd.DataFrame([{
    "open_facilities": ", ".join(ufl_solution["open_facilities"]),
    "n_open": ufl_solution["n_open"],
    "mean_assignment_km": ufl_solution["mean_km"],
    "fallback_assignments": ufl_solution["fallback_assignments"],
    "objective": ufl_solution["objective"],
}])

ufl_summary, assignment.sort_values(["source_id", "target_id"]), ufl_sensitivity.head(8)

## Model 2: Assigned-School Routing

After facility location, the slide deck solves a TSP for each selected facility and its assigned schools. The demonstration below solves each miniature assigned route exactly with dynamic programming. It uses road-network shortest paths where available and keeps a fallback flag where a segment crosses a disconnected component.

This follows the slide interpretation: route figures are diagnostic objects. A route can be mathematically complete while still revealing that the road graph, geocoding, or component assignment needs review.

In [ ]:
G = nx.DiGraph()
for row in nodes.itertuples(index=False):
    G.add_node(row.node_id, lon=row.lon, lat=row.lat)
for row in edges.itertuples(index=False):
    G.add_edge(row.u, row.v, length_m=row.length_m)

node_coords = nodes.set_index("node_id")[["lon", "lat"]].to_dict("index")

def snapped_point_table(facility_id, assigned_targets):
    facility = snapped_candidates[snapped_candidates["source_id"] == facility_id].rename(columns={"source_id": "point_id"})
    facility = facility[["point_id", "lon", "lat", "nearest_node"]].assign(kind="facility")
    target_rows = snapped_schools[snapped_schools["target_id"].isin(assigned_targets)].rename(columns={"target_id": "point_id"})
    target_rows = target_rows[["point_id", "lon", "lat", "nearest_node"]].assign(kind="school")
    return pd.concat([facility, target_rows], ignore_index=True)

def pair_distance(a, b):
    try:
        dist = nx.shortest_path_length(G, a.nearest_node, b.nearest_node, weight="length_m")
        return float(dist), False, "road_network"
    except nx.NetworkXNoPath:
        return straight_line_m(a.lon, a.lat, b.lon, b.lat), True, "straight_line_disconnected_component"

def tsp_dynamic(points):
    ids = points["point_id"].tolist()
    depot = ids[0]
    schools_only = ids[1:]
    pair = {}
    fallback_pair = {}
    point_by_id = {r.point_id: r for r in points.itertuples(index=False)}
    for a in ids:
        for b in ids:
            if a == b:
                continue
            dist, fallback, mode = pair_distance(point_by_id[a], point_by_id[b])
            pair[(a, b)] = dist
            fallback_pair[(a, b)] = fallback

    @lru_cache(None)
    def dp(current, remaining):
        remaining = tuple(remaining)
        if not remaining:
            return pair[(current, depot)], [depot]
        best_cost = float("inf")
        best_path = None
        for nxt in remaining:
            next_remaining = tuple(x for x in remaining if x != nxt)
            tail_cost, tail_path = dp(nxt, next_remaining)
            cost = pair[(current, nxt)] + tail_cost
            if cost < best_cost:
                best_cost = cost
                best_path = [nxt] + tail_path
        return best_cost, best_path

    total, path_tail = dp(depot, tuple(schools_only))
    route = [depot] + path_tail
    legs = []
    for a, b in zip(route[:-1], route[1:]):
        legs.append({"from": a, "to": b, "distance_m": pair[(a, b)], "fallback": fallback_pair[(a, b)]})
    return route, pd.DataFrame(legs)

route_outputs = {}
for facility_id, group in assignment.groupby("source_id"):
    pts = snapped_point_table(facility_id, group["target_id"].tolist())
    route, legs = tsp_dynamic(pts)
    route_outputs[facility_id] = {"points": pts, "route": route, "legs": legs}

route_summary = pd.DataFrame([
    {
        "facility": facility_id,
        "assigned_schools": int((out["points"]["kind"] == "school").sum()),
        "route_km": out["legs"]["distance_m"].sum() / 1000.0,
        "fallback_segments": int(out["legs"]["fallback"].sum()),
        "route_order": " -> ".join(out["route"]),
    }
    for facility_id, out in route_outputs.items()
]).sort_values("facility")

route_summary

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.8))
for row in edges.iloc[::2].itertuples(index=False):
    a = node_coords[row.u]
    b = node_coords[row.v]
    ax.plot([a["lon"], b["lon"]], [a["lat"], b["lat"]], color="#9aa5b1", linewidth=2, zorder=1)

colors = {facility: color for facility, color in zip(sorted(assignment["source_id"].unique()), ["#1b6ca8", "#d64545", "#2f855a", "#805ad5"])}
for facility_id, group in assignment.groupby("source_id"):
    ax.scatter(group["lon"], group["lat"], s=group["demand"] * 1.2, color=colors[facility_id], alpha=0.55, edgecolor="white", label=f"Schools assigned to {facility_id}", zorder=3)

selected_candidates = candidates[candidates["source_id"].isin(ufl_solution["open_facilities"])]
not_selected = candidates[~candidates["source_id"].isin(ufl_solution["open_facilities"])]
ax.scatter(not_selected["lon"], not_selected["lat"], marker="^", s=110, color="#8d99ae", edgecolor="white", label="Closed candidates", zorder=4)
ax.scatter(selected_candidates["lon"], selected_candidates["lat"], marker="^", s=190, color=[colors[x] for x in selected_candidates["source_id"]], edgecolor="black", label="Open facilities", zorder=5)
for row in selected_candidates.itertuples(index=False):
    ax.text(row.lon, row.lat + 0.18, row.source_id, ha="center", fontsize=9, weight="bold")
for row in schools.itertuples(index=False):
    ax.text(row.lon + 0.05, row.lat - 0.08, row.target_id[-3:], fontsize=7)
ax.set_title("Miniature Sabu facility-location result using road/fallback distances", fontsize=13, weight="bold")
ax.set_xlabel("Synthetic longitude")
ax.set_ylabel("Synthetic latitude")
ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1), frameon=False)
ax.grid(True, color="#e6e8e6")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.8))
for row in edges.iloc[::2].itertuples(index=False):
    a = node_coords[row.u]
    b = node_coords[row.v]
    ax.plot([a["lon"], b["lon"]], [a["lat"], b["lat"]], color="#c4c9cf", linewidth=2, zorder=1)

point_locations = {}
for row in candidates.itertuples(index=False):
    point_locations[row.source_id] = (row.lon, row.lat)
for row in schools.itertuples(index=False):
    point_locations[row.target_id] = (row.lon, row.lat)

for facility_id, out in route_outputs.items():
    color = colors[facility_id]
    for leg in out["legs"].itertuples(index=False):
        a = point_locations[leg._0] if hasattr(leg, "_0") else point_locations[getattr(leg, "from")]
        b = point_locations[leg.to]
        linestyle = "--" if leg.fallback else "-"
        linewidth = 2.8 if leg.fallback else 1.8
        ax.plot([a[0], b[0]], [a[1], b[1]], color=color, linestyle=linestyle, linewidth=linewidth, alpha=0.88, zorder=2)

for facility_id, group in assignment.groupby("source_id"):
    ax.scatter(group["lon"], group["lat"], s=group["demand"] * 1.05, color=colors[facility_id], alpha=0.65, edgecolor="white", zorder=3)
ax.scatter(selected_candidates["lon"], selected_candidates["lat"], marker="^", s=190, color=[colors[x] for x in selected_candidates["source_id"]], edgecolor="black", zorder=5)
for row in selected_candidates.itertuples(index=False):
    ax.text(row.lon, row.lat + 0.18, row.source_id, ha="center", fontsize=9, weight="bold")
ax.plot([], [], color="#333333", linestyle="-", label="Road-distance TSP leg")
ax.plot([], [], color="#333333", linestyle="--", label="Straight-line fallback leg")
ax.set_title("Assigned-school TSP diagnostics: fallback legs expose disconnected components", fontsize=13, weight="bold")
ax.set_xlabel("Synthetic longitude")
ax.set_ylabel("Synthetic latitude")
ax.legend(loc="upper left", bbox_to_anchor=(1.02, 1), frameon=False)
ax.grid(True, color="#e6e8e6")
ax.spines[["top", "right"]].set_visible(False)
fig.tight_layout()
plt.show()

## Comparison With The Slide Experiment

The notebook-scale run intentionally has only a dozen schools, but it follows the slide experiment's structure:

- It solves a fixed-charge uncapacitated facility-location problem over road-based distances.
- The fixed opening cost is calibrated as `2 km * total demand`, as in the slides.
- It assigns each school to one selected facility.
- It then solves one TSP per selected facility's assigned schools.
- It keeps straight-line fallback legs visible when the road network is disconnected.

The slide-scale Sabu run selected 2 facilities for 303 schools, with routes of 151.57 km and 258.40 km. This miniature run is a development and documentation fixture: once the real NTT school table, candidate list, island boundaries, and cached road matrix are available in the shared cache, the same contracts can run the full experiment.

In [ ]:
comparison = pd.DataFrame([
    {"step": "Geocoding coverage", "slide_scale": "14,101 / 14,103 geocoded", "notebook_recreation": "Uses already selected synthetic routing targets"},
    {"step": "Facility location", "slide_scale": "27 facilities for 14,099 served schools; Sabu 2 / 303", "notebook_recreation": f"{ufl_solution['n_open']} facilities for {len(schools)} schools"},
    {"step": "Routing", "slide_scale": "Sabu routes 151.57 km and 258.40 km", "notebook_recreation": "; ".join(f"{r.facility}: {r.route_km:.2f} km" for r in route_summary.itertuples(index=False))},
    {"step": "Disconnected components", "slide_scale": "4 straight-line fallback segments in one Sabu route", "notebook_recreation": f"{route_summary['fallback_segments'].sum()} fallback segment(s) flagged"},
])
comparison